# 01 · 探索性数据分析 (EDA)

**输入文件**：`../data/train.csv`  
**输出文件**：`../outputs/figures/01_*.png`、`../outputs/tables/01_feature_stats.csv`  
**随机种子**：42

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

SEED = 42
np.random.seed(SEED)

FIG_DIR = Path('../outputs/figures')
TAB_DIR = Path('../outputs/tables')
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

## 1. 读取数据 & 样本规模统计

In [2]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

feature_cols = [c for c in train.columns if c.startswith('var_')]

n_train   = len(train)
n_test    = len(test)
n_feat    = len(feature_cols)
n_pos     = int(train['target'].sum())
n_neg     = n_train - n_pos
pos_ratio = n_pos / n_train

print(f'训练集样本数 : {n_train:,}')
print(f'测试集样本数 : {n_test:,}')
print(f'特征数量     : {n_feat}')
print(f'正例 (target=1) : {n_pos:,}  ({pos_ratio:.2%})')
print(f'负例 (target=0) : {n_neg:,}  ({1-pos_ratio:.2%})')
print(f'\n缺失值 (train) : {train.isnull().sum().sum()}')
print(f'缺失值 (test)  : {test.isnull().sum().sum()}')

训练集样本数 : 200,000
测试集样本数 : 200,000
特征数量     : 200
正例 (target=1) : 20,098  (10.05%)
负例 (target=0) : 179,902  (89.95%)

缺失值 (train) : 0
缺失值 (test)  : 0


## 2. target 类别分布

In [3]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = train['target'].value_counts().sort_index()
bars = ax.bar(['负例 (0)', '正例 (1)'], counts.values,
              color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.5)
for bar, cnt in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'{cnt:,}\n({cnt/n_train:.1%})', ha='center', va='bottom', fontsize=10)
ax.set_title('Target 类别分布', fontsize=13)
ax.set_ylabel('样本数')
ax.set_ylim(0, n_neg * 1.15)
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '01_target_distribution.png')
plt.show()
print('saved: 01_target_distribution.png')

saved: 01_target_distribution.png


## 3. 特征统计表（均值、标准差、偏度、峰度、缺失值）

In [4]:
X = train[feature_cols]
y = train['target']

stats = pd.DataFrame({
    'mean'     : X.mean(),
    'std'      : X.std(),
    'min'      : X.min(),
    'max'      : X.max(),
    'skewness' : X.skew(),
    'kurtosis' : X.kurtosis(),
    'missing'  : X.isnull().sum(),
})

stats.to_csv(TAB_DIR / '01_feature_stats.csv')
print(f'特征统计表已保存 ({len(stats)} 行)')
stats.head(10)

特征统计表已保存 (200 行)


,mean,std,min,max,skewness,kurtosis,missing
var_0,10.679914,3.040051,0.4084,20.3150,0.235639,-0.273593,0
var_1,-1.627622,4.050044,-15.0434,10.3768,0.053115,-0.607265,0
var_2,10.715192,2.640894,2.1171,19.3530,0.260313,-0.336616,0
var_3,6.796529,2.043319,-0.0402,13.1883,-0.003548,-0.602623,0
var_4,11.078333,1.623150,5.0748,16.6714,-0.048210,-0.534993,0
var_5,-5.065317,7.863267,-32.5626,17.2516,-0.002038,-0.668954,0
var_6,5.408949,0.866607,2.3473,8.4477,0.149476,-0.383997,0
var_7,16.545850,3.418076,5.3497,27.6918,0.084598,-0.671014,0
var_8,0.284162,3.332634,-10.5055,10.1513,-0.104643,-0.803599,0
var_9,7.567236,1.235070,3.9705,11.1506,-0.175433,-0.760764,0


## 4. 正负样本均值差异 Top 20

In [5]:
mean_pos = X[y == 1].mean()
mean_neg = X[y == 0].mean()
mean_diff = (mean_pos - mean_neg).abs().sort_values(ascending=False)
top20_feats = mean_diff.head(20).index.tolist()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(mean_diff.head(20).index[::-1], mean_diff.head(20).values[::-1],
        color='#4C72B0', edgecolor='white')
ax.set_xlabel('|mean_positive - mean_negative|')
ax.set_title('正负样本均值差异 Top 20 特征')
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '01_mean_diff_top20.png')
plt.show()
print('saved: 01_mean_diff_top20.png')

saved: 01_mean_diff_top20.png


## 5. Top 20 特征 KDE 分布（正负样本对比）

In [6]:
fig, axes = plt.subplots(4, 5, figsize=(22, 16))
for ax, feat in zip(axes.flatten(), top20_feats):
    for label, color, lbl in [(0, '#4C72B0', 'target=0'), (1, '#DD8452', 'target=1')]:
        vals = X.loc[y == label, feat].dropna()
        sns.kdeplot(vals, ax=ax, color=color, label=lbl, fill=True, alpha=0.35)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
    ax.legend(fontsize=7)
plt.suptitle('Top 20 高区分度特征 KDE 分布（正负样本对比）', fontsize=14, y=1.01)
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '01_kde_top20.png', bbox_inches='tight')
plt.show()
print('saved: 01_kde_top20.png')

saved: 01_kde_top20.png


## 6. 与 target 相关性 Top 30 热力图

In [7]:
corr_with_target = X.corrwith(y).abs().sort_values(ascending=False)
top30_feats = corr_with_target.head(30).index.tolist()

# 热力图
corr_matrix = train[top30_feats + ['target']].corr()
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_matrix, mask=mask, ax=ax, cmap='coolwarm', center=0,
            square=True, linewidths=0.3, cbar_kws={'shrink': 0.7},
            xticklabels=True, yticklabels=True)
ax.set_title('与 target 相关性最高 Top 30 特征相关系数热力图', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / '01_corr_heatmap_top30.png', bbox_inches='tight')
plt.show()
print('saved: 01_corr_heatmap_top30.png')

# 柱状图
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(corr_with_target.head(20).index[::-1], corr_with_target.head(20).values[::-1],
        color='#55A868', edgecolor='white')
ax.set_xlabel('|Pearson 相关系数|')
ax.set_title('与 target 相关性 Top 20 特征')
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '01_corr_top20_bar.png')
plt.show()
print('saved: 01_corr_top20_bar.png')

saved: 01_corr_heatmap_top30.png
saved: 01_corr_top20_bar.png


## 7. 单变量分箱正例率折线图（Top 10 高区分度特征）

In [8]:
top10_feats = mean_diff.head(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, feat in zip(axes.flatten(), top10_feats):
    df_plot = train[[feat, 'target']].copy()
    df_plot['bin'] = pd.qcut(df_plot[feat], q=5, duplicates='drop')
    pos_rate = df_plot.groupby('bin', observed=True)['target'].mean()
    ax.plot(range(len(pos_rate)), pos_rate.values, marker='o', color='#DD8452', linewidth=2)
    ax.axhline(y=pos_ratio, color='gray', linestyle='--', alpha=0.6, label=f'全局均值 {pos_ratio:.2%}')
    ax.set_title(feat, fontsize=9)
    ax.set_ylabel('正例率')
    ax.set_xlabel('分箱（低→高）')
    ax.set_xticks(range(len(pos_rate)))
    ax.set_xticklabels([f'Q{i+1}' for i in range(len(pos_rate))], fontsize=8)
    ax.legend(fontsize=7)

plt.suptitle('Top 10 特征等频分箱（5档）正例率折线图', fontsize=14, y=1.01)
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '01_binning_positive_rate.png', bbox_inches='tight')
plt.show()
print('saved: 01_binning_positive_rate.png')

saved: 01_binning_positive_rate.png


## 8. WoE / IV 分析（Top 20 IV 特征）

In [9]:
def calc_iv(series, target, n_bins=5):
    """计算单变量 IV，等频分箱"""
    df = pd.DataFrame({'x': series, 'y': target})
    df['bin'] = pd.qcut(df['x'], q=n_bins, duplicates='drop')
    total_pos = target.sum()
    total_neg = len(target) - total_pos
    rows = []
    for bin_val, grp in df.groupby('bin', observed=True):
        p = grp['y'].sum() / total_pos
        n = (len(grp) - grp['y'].sum()) / total_neg
        p = max(p, 1e-9)
        n = max(n, 1e-9)
        woe = np.log(p / n)
        iv  = (p - n) * woe
        rows.append({'bin': str(bin_val), 'pos_rate': grp['y'].mean(),
                     'woe': woe, 'iv': iv})
    return pd.DataFrame(rows), sum(r['iv'] for r in rows)

iv_records = []
for feat in feature_cols:
    _, iv = calc_iv(X[feat], y)
    iv_records.append({'feature': feat, 'iv': iv})

iv_df = pd.DataFrame(iv_records).sort_values('iv', ascending=False).reset_index(drop=True)

def iv_label(iv):
    if iv < 0.02:  return '无用'
    if iv < 0.1:   return '弱'
    if iv < 0.3:   return '中等'
    return '强'

iv_df['预测能力'] = iv_df['iv'].map(iv_label)
iv_df.head(20).to_csv(TAB_DIR / 'woe_iv_top20.csv', index=False)
print('Top 20 IV 特征：')
iv_df.head(20)

Top 20 IV 特征：


,feature,iv,预测能力
0,var_81,0.076808,弱
1,var_139,0.061793,弱
2,var_12,0.049324,弱
3,var_110,0.048496,弱
4,var_6,0.046555,弱
5,var_146,0.042271,弱
6,var_26,0.040333,弱
7,var_76,0.038508,弱
8,var_53,0.037226,弱
9,var_148,0.036547,弱


In [10]:
fig, ax = plt.subplots(figsize=(10, 5))
top20_iv = iv_df.head(20)
colors = ['#DD8452' if v == '强' else '#4C72B0' if v == '中等' else '#55A868'
          for v in top20_iv['预测能力']]
ax.barh(top20_iv['feature'][::-1], top20_iv['iv'][::-1], color=colors[::-1], edgecolor='white')
ax.axvline(x=0.1, color='gray', linestyle='--', alpha=0.7, label='IV=0.1 (弱/中分界)')
ax.axvline(x=0.3, color='red',  linestyle='--', alpha=0.5, label='IV=0.3 (中/强分界)')
ax.set_xlabel('IV (Information Value)')
ax.set_title('Top 20 特征 IV 值（预测能力）')
ax.legend(fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / '01_iv_top20.png')
plt.show()
print('saved: 01_iv_top20.png')

saved: 01_iv_top20.png


## 小结

- 训练集 200,000 条，200 个匿名数值特征，**无缺失值**
- 正例约占 10%，存在明显类别不平衡，主指标应使用 ROC-AUC / PR-AUC
- 正负样本均值差异最大的特征（如 var_81、var_139 等）在 KDE 图中呈现明显分布偏移，具备一定预测信号
- 分箱正例率折线图显示部分特征具有单调趋势，与 target 存在非线性正/负相关
- WoE/IV 分析表明约有 20+ 个特征 IV > 0.1（中等及以上预测能力）
- 特征全部匿名，缺乏业务语义，解释性受限——这将在报告局限性章节中说明